# Get data for New Mexico for Power BI

Goals
* get rsinventory query data (ie folder of parquet files for AOI) but from "dev_riverscapes"."materialized_dgo_grazing_nm" and including new `graz_globalid` column
* get aoi query of default.blm_natl_grazing_allotments


In [ ]:
# Common Imports and Setup 
from pathlib import Path

import pandas as pd
import geopandas as gpd
import plotly.express as px
import matplotlib.pyplot as plt

from rsxml import Logger

# Import from src
# These imports will work once the path is set up above
from util import prepare_gdf_for_athena
from util.athena import (query_to_dataframe,
                          get_field_metadata,
                          aoi_query_to_local_parquet,
                          query_to_local_parquet,
                          )
from util.athena.athena import prepare_aoi_query  
# this is only needed if the util code has changed in middle of our session
# import importlib
# import util.pandas.pandas_utilities
# importlib.reload(util.pandas.pandas_utilities)
from util.pandas.pandas_utilities import pprint_df_meta
from util.pandas import RSFieldMeta

# Set pandas options for better display
pd.set_option('display.max_columns', None)


In [ ]:
# Global Parameters for this NB
output_dir = Path(r'C:\nardata\work\reporting\lascruces-sra\notebook_data')
path_to_shape = Path(r'C:\nardata\localcode\rs-reports-gen\src\reports\rpt_riverscapes_inventory\example\las_cruces_office.geojson')

In [ ]:
def get_dgo_pq(report_dir: Path):
    log = Logger('get dgo pq')
    # load shape as gdf
    aoi_gdf = gpd.read_file(path_to_shape)

    fields_we_need = "level_path, seg_distance, centerline_length, segment_area, fcode, fcode_desc, longitude, latitude, ownership, ownership_desc, state, county, drainage_area, stream_name, stream_order, stream_length, huc12, rel_flow_length, channel_area, integrated_width, low_lying_ratio, elevated_ratio, floodplain_ratio, acres_vb_per_mile, hect_vb_per_km, channel_width, lf_agriculture_prop, lf_agriculture, lf_developed_prop, lf_developed, lf_riparian_prop, lf_riparian, ex_riparian, hist_riparian, prop_riparian, hist_prop_riparian, develop, road_len, road_dens, rail_len, rail_dens, land_use_intens, road_dist, rail_dist, div_dist, canal_dist, infra_dist, fldpln_access, access_fldpln_extent, confinement_ratio, brat_capacity,brat_hist_capacity, riparian_veg_departure, riparian_condition, rme_project_id, rme_project_name, graz_globalid"
    query_str = f"SELECT {fields_we_need} FROM dev_riverscapes.materialized_rpt_rme_grazing_nm WHERE {{prefilter_condition}} AND {{intersects_condition}}"

    query_gdf, simplification_results = prepare_gdf_for_athena(aoi_gdf)
    if not simplification_results.success:
        raise ValueError("Unable to simplify input geometry sufficiently to insert into Athena query")
    if simplification_results.simplified:
        log.warning(
            f"""Input polygon was simplified using tolerance of {simplification_results.tolerance_m} metres for the purpose of intersecting with DGO geometries in the database.
            If you require a higher precision extract, please contact support@riverscapes.freshdesk.com.""")

    log.info("Querying Athena for data for AOI")
    parquet_data_source = report_dir / "dgo_pq"

    aoi_query_to_local_parquet(
            query_str,
            geometry_field_expression='ST_GeomFromBinary(dgo_geom)',
            geom_bbox_field='dgo_geom_bbox',
            aoi_gdf=query_gdf,
            local_path=parquet_data_source
        )

In [ ]:
# get_dgo_pq(output_dir)
# DONE:  [AOI Query to Local PQ] Successfully unloaded data for AOI to C:\nardata\work\reporting\lascruces-sra\notebook_data\dgo_pq

In [ ]:
def get_grazing_areas(report_dir:Path):
    log = Logger('get grazing pq')
    # load shape as gdf
    aoi_gdf = gpd.read_file(path_to_shape)

    fields_we_need = "allot_no, allot_name, admin_st, adm_ofc_cd, st_allot, globalid"
    query_str = f"SELECT {fields_we_need} FROM default.blm_natl_grazing_allotments WHERE {{prefilter_condition}} AND {{intersects_condition}}"

    query_gdf, simplification_results = prepare_gdf_for_athena(aoi_gdf)
    if not simplification_results.success:
        raise ValueError("Unable to simplify input geometry sufficiently to insert into Athena query")
    if simplification_results.simplified:
        log.warning(
            f"""Input polygon was simplified using tolerance of {simplification_results.tolerance_m} metres for the purpose of intersecting with DGO geometries in the database.
            If you require a higher precision extract, please contact support@riverscapes.freshdesk.com.""")

    log.info("Querying Athena for data for AOI")
    parquet_data_source = report_dir / "grazing_pq"

    aoi_query_to_local_parquet(
            query_str,
            geometry_field_expression='ST_GeomFromBinary(geometry)',
            geom_bbox_field='geometry_bbox',
            aoi_gdf=query_gdf,
            local_path=parquet_data_source
        )

In [ ]:
# get_grazing_areas(output_dir)
# [INFO] [AOI Query to Local PQ] Successfully unloaded data for AOI to C:\nardata\work\reporting\lascruces-sra\notebook_data\grazing_pq

In [ ]:
def get_huc_data(parquet_data_folder: Path):
    log = Logger('get HUC pq')

    # this is a new type of intersection query that includes the percent area of intersection
    fields_we_need = "huc10.huc10 AS huc, rscontext.project_id, rscontext.hucname, rscontext.hucareasqkm, dem_bins, ST_GeomFromBinary(geometry)"
    query_str = f"""WITH input_geom AS (SELECT {{aoi_geom_str}} AS geom)
    SELECT {fields_we_need} 
    FROM input_geom, (
    SELECT ) 
    wbdhu10_cleaned huc10 LEFT JOIN rs_context_huc10 rscontext ON huc10.huc10 = rscontext.huc
    WHERE {{prefilter_condition}} AND {{intersects_condition}}"""


    # load shape as gdf
    aoi_gdf = gpd.read_file(path_to_shape)

    # simiplify gdf if needed
    query_gdf, simplification_results = prepare_gdf_for_athena(aoi_gdf)
    if not simplification_results.success:
        raise ValueError("Unable to simplify input geometry sufficiently to insert into Athena query")
    if simplification_results.simplified:
        log.warning(
            f"""Input polygon was simplified using tolerance of {simplification_results.tolerance_m} metres for the purpose of intersecting with DGO geometries in the database.
            If you require a higher precision extract, please contact support@riverscapes.freshdesk.com.""")

    

    log.info("Querying Athena for data for AOI")
    
    
    if not aoi_geom_str:
        raise ValueError("AOI geometry exceeds Athena size limit. Simplify and try again.")


    # query_to_local_parquet(query=
    #                        local_path=parquet_data_folder)

    aoi_query_to_local_parquet(
            query_str,
            geometry_field_expression='ST_GeomFromBinary(geometry)',
            geom_bbox_field='geometry_bbox',
            aoi_gdf=query_gdf,
            local_path=parquet_data_folder
        )

In [ ]:
# get_huc_data(output_dir / "huc")
# [AOI Query to Local PQ] Successfully unloaded data for AOI to C:\nardata\work\reporting\lascruces-sra\notebook_data\huc